# Cross-Encoder v0.7 — Phase 1b: LR Grid Search (RoBERTa-base)

## Bối cảnh
Attempt đầu (`ms-marco-electra-base`) thất bại hoàn toàn: LabelAcc ~25-27% (thua v0.6 ~39pp).  
ELECTRA là discriminator architecture — không tương thích tốt với CrossEncoder Sigmoid+MSE fine-tuning.

## Hypothesis
Thay bằng `cross-encoder/ms-marco-roberta-base` (768-dim, RoBERTa encoder, MS MARCO pre-trained):
- RoBERTa encoder tương thích tốt với CrossEncoder fine-tuning (cùng paradigm với MiniLM)
- Hidden dim 768 vs 384 của MiniLM-L12 → representation giàu hơn
- Đã được pre-train trực tiếp trên MS MARCO ranking task → không bị catastrophic như ELECTRA

## LR Grid
| LR | Lý do |
|---|---|
| 2e-5 | Conservative — chuẩn RoBERTa fine-tune |
| 3e-5 | Middle ground |
| 5e-5 | Aggressive — optimal MiniLM ở v0.6, thử xem RoBERTa có chịu không |

## v0.6 Baseline
| | Best single | Ensemble |
|---|---|---|
| LabelAcc | 64.95% | **65.80%** |
| MAE | 9.27 | 9.11 |

## v0.7 ELECTRA (thất bại)
| LR | Test LabelAcc | vs v0.6 |
|---|---|---|
| 2e-5 | 25.60% | -39.35pp |
| 3e-5 | 26.05% | -38.90pp |
| 5e-5 | 24.70% | -40.25pp |

## Output
- Report mỗi LR: `artifacts/reports/fine_tune_cross_encoder_v0.7_roberta_lr{LR}_report.json`
- Summary: `artifacts/reports/fine_tune_cross_encoder_v0.7_roberta_lr_grid_search_report.json`
- Dùng best LR cho Phase 2 (Ensemble)

In [ ]:
# === CELL 1: Clone & Install ===
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.7
!git pull origin experiment/cross-encoder-v0.7
!pip install -q -r requirements.txt

import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')

In [ ]:
# === CELL 2: Load Data & Helpers ===
import json
import numpy as np
import random
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

BASE_MODEL = 'cross-encoder/ms-marco-roberta-base'
DATA_DIR   = 'datasets/versions/v0.5/cross_encoder'

V06_BEST_SINGLE  = 0.6495  # lr5e-05-15ep
V06_ENSEMBLE_ACC = 0.6580


class CECorrelationEvaluator:
    def __init__(self, sentence_pairs, labels_0_1):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1     = labels_0_1

    @classmethod
    def from_input_examples(cls, examples):
        return cls([ex.texts for ex in examples], [ex.label for ex in examples])

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0


def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]


def to_examples(records):
    return [InputExample(texts=[r['cv_text'], r['jd_text']], label=r['label']) for r in records]


def compute_metrics(model, examples):
    preds      = model.predict([ex.texts for ex in examples], batch_size=32, show_progress_bar=False)
    preds_100  = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])
    return {
        'LabelAcc': float(np.mean(np.abs(preds_100 - labels_100) <= 10)),
        'MAE':      float(np.mean(np.abs(preds_100 - labels_100))),
        'RMSE':     float(np.sqrt(np.mean((preds_100 - labels_100) ** 2))),
    }


train_examples = to_examples(load_jsonl(f'{DATA_DIR}/cross_encoder_train.jsonl'))
val_examples   = to_examples(load_jsonl(f'{DATA_DIR}/cross_encoder_validation.jsonl'))
test_examples  = to_examples(load_jsonl(f'{DATA_DIR}/cross_encoder_test.jsonl'))

print(f'Data: {len(train_examples)} train | {len(val_examples)} val | {len(test_examples)} test')
print(f'Model: {BASE_MODEL}')

In [ ]:
# === CELL 3: LR = 2e-05 ===
# Chạy độc lập — kết quả được save Drive ngay sau khi xong
import os, json, shutil, torch, numpy as np, random
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')

LR         = 2e-5
SEED       = 42
EPOCHS     = 10
BATCH_SIZE = 16

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

run_name     = f'v0.7-roberta-lr{LR:.0e}-seed{SEED}'
output_dir   = f'artifacts/models/cross-encoder-cv-jd-{run_name}'
total_steps  = (len(train_examples) // BATCH_SIZE + 1) * EPOCHS
warmup_steps = int(total_steps * 0.1)

print(f'LR={LR:.0e} | Epochs={EPOCHS} | Warmup={warmup_steps} | Model={BASE_MODEL}')

model      = CrossEncoder(BASE_MODEL, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator  = CECorrelationEvaluator.from_input_examples(val_examples)
dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_dataloader=dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': LR},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True,
)

best_model = CrossEncoder(output_dir)
val_m  = compute_metrics(best_model, val_examples)
test_m = compute_metrics(best_model, test_examples)

print(f'\nLR={LR:.0e} done')
print(f'  Val  LabelAcc: {val_m["LabelAcc"]*100:.2f}%  | MAE: {val_m["MAE"]:.2f}  | RMSE: {val_m["RMSE"]:.2f}')
print(f'  Test LabelAcc: {test_m["LabelAcc"]*100:.2f}%  | MAE: {test_m["MAE"]:.2f}  | RMSE: {test_m["RMSE"]:.2f}  ({(test_m["LabelAcc"]-V06_BEST_SINGLE)*100:+.2f}pp vs v0.6)')

result = {
    'learning_rate': LR, 'run': run_name, 'epochs': EPOCHS,
    'val':  {'LabelAcc': val_m['LabelAcc'], 'MAE': val_m['MAE'], 'RMSE': val_m['RMSE']},
    'test': {'LabelAcc': test_m['LabelAcc'], 'MAE': test_m['MAE'], 'RMSE': test_m['RMSE']},
}

os.makedirs('artifacts/reports', exist_ok=True)
report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.7_roberta_lr{LR:.0e}_report.json'
with open(report_path, 'w') as f:
    json.dump(result, f, indent=2)

try:
    DRIVE_MODELS  = '/content/drive/MyDrive/ai-recruiter/models'
    DRIVE_REPORTS = '/content/drive/MyDrive/ai-recruiter/reports'
    os.makedirs(DRIVE_MODELS,  exist_ok=True)
    os.makedirs(DRIVE_REPORTS, exist_ok=True)

    dest = f'{DRIVE_MODELS}/{run_name}'
    if os.path.exists(dest): shutil.rmtree(dest)
    shutil.copytree(output_dir, dest)
    shutil.copy(report_path, f'{DRIVE_REPORTS}/{os.path.basename(report_path)}')
    print(f'  Model  → Drive ✓  ({dest})')
    print(f'  Report → Drive ✓  ({DRIVE_REPORTS}/{os.path.basename(report_path)})')
except Exception as e:
    print(f'  Drive upload skipped: {e}')

In [ ]:
# === CELL 4: LR = 3e-05 ===
# Chạy độc lập — kết quả được save Drive ngay sau khi xong
import os, json, shutil, torch, numpy as np, random
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')

LR         = 3e-5
SEED       = 42
EPOCHS     = 10
BATCH_SIZE = 16

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

run_name     = f'v0.7-roberta-lr{LR:.0e}-seed{SEED}'
output_dir   = f'artifacts/models/cross-encoder-cv-jd-{run_name}'
total_steps  = (len(train_examples) // BATCH_SIZE + 1) * EPOCHS
warmup_steps = int(total_steps * 0.1)

print(f'LR={LR:.0e} | Epochs={EPOCHS} | Warmup={warmup_steps} | Model={BASE_MODEL}')

model      = CrossEncoder(BASE_MODEL, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator  = CECorrelationEvaluator.from_input_examples(val_examples)
dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_dataloader=dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': LR},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True,
)

best_model = CrossEncoder(output_dir)
val_m  = compute_metrics(best_model, val_examples)
test_m = compute_metrics(best_model, test_examples)

print(f'\nLR={LR:.0e} done')
print(f'  Val  LabelAcc: {val_m["LabelAcc"]*100:.2f}%  | MAE: {val_m["MAE"]:.2f}  | RMSE: {val_m["RMSE"]:.2f}')
print(f'  Test LabelAcc: {test_m["LabelAcc"]*100:.2f}%  | MAE: {test_m["MAE"]:.2f}  | RMSE: {test_m["RMSE"]:.2f}  ({(test_m["LabelAcc"]-V06_BEST_SINGLE)*100:+.2f}pp vs v0.6)')

result = {
    'learning_rate': LR, 'run': run_name, 'epochs': EPOCHS,
    'val':  {'LabelAcc': val_m['LabelAcc'], 'MAE': val_m['MAE'], 'RMSE': val_m['RMSE']},
    'test': {'LabelAcc': test_m['LabelAcc'], 'MAE': test_m['MAE'], 'RMSE': test_m['RMSE']},
}

os.makedirs('artifacts/reports', exist_ok=True)
report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.7_roberta_lr{LR:.0e}_report.json'
with open(report_path, 'w') as f:
    json.dump(result, f, indent=2)

try:
    DRIVE_MODELS  = '/content/drive/MyDrive/ai-recruiter/models'
    DRIVE_REPORTS = '/content/drive/MyDrive/ai-recruiter/reports'
    os.makedirs(DRIVE_MODELS,  exist_ok=True)
    os.makedirs(DRIVE_REPORTS, exist_ok=True)

    dest = f'{DRIVE_MODELS}/{run_name}'
    if os.path.exists(dest): shutil.rmtree(dest)
    shutil.copytree(output_dir, dest)
    shutil.copy(report_path, f'{DRIVE_REPORTS}/{os.path.basename(report_path)}')
    print(f'  Model  → Drive ✓  ({dest})')
    print(f'  Report → Drive ✓  ({DRIVE_REPORTS}/{os.path.basename(report_path)})')
except Exception as e:
    print(f'  Drive upload skipped: {e}')

In [ ]:
# === CELL 5: LR = 5e-05 ===
# Chạy độc lập — kết quả được save Drive ngay sau khi xong
import os, json, shutil, torch, numpy as np, random
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')

LR         = 5e-5
SEED       = 42
EPOCHS     = 10
BATCH_SIZE = 16

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

run_name     = f'v0.7-roberta-lr{LR:.0e}-seed{SEED}'
output_dir   = f'artifacts/models/cross-encoder-cv-jd-{run_name}'
total_steps  = (len(train_examples) // BATCH_SIZE + 1) * EPOCHS
warmup_steps = int(total_steps * 0.1)

print(f'LR={LR:.0e} | Epochs={EPOCHS} | Warmup={warmup_steps} | Model={BASE_MODEL}')

model      = CrossEncoder(BASE_MODEL, num_labels=1, default_activation_function=torch.nn.Sigmoid())
evaluator  = CECorrelationEvaluator.from_input_examples(val_examples)
dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_dataloader=dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': LR},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True,
)

best_model = CrossEncoder(output_dir)
val_m  = compute_metrics(best_model, val_examples)
test_m = compute_metrics(best_model, test_examples)

print(f'\nLR={LR:.0e} done')
print(f'  Val  LabelAcc: {val_m["LabelAcc"]*100:.2f}%  | MAE: {val_m["MAE"]:.2f}  | RMSE: {val_m["RMSE"]:.2f}')
print(f'  Test LabelAcc: {test_m["LabelAcc"]*100:.2f}%  | MAE: {test_m["MAE"]:.2f}  | RMSE: {test_m["RMSE"]:.2f}  ({(test_m["LabelAcc"]-V06_BEST_SINGLE)*100:+.2f}pp vs v0.6)')

result = {
    'learning_rate': LR, 'run': run_name, 'epochs': EPOCHS,
    'val':  {'LabelAcc': val_m['LabelAcc'], 'MAE': val_m['MAE'], 'RMSE': val_m['RMSE']},
    'test': {'LabelAcc': test_m['LabelAcc'], 'MAE': test_m['MAE'], 'RMSE': test_m['RMSE']},
}

os.makedirs('artifacts/reports', exist_ok=True)
report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.7_roberta_lr{LR:.0e}_report.json'
with open(report_path, 'w') as f:
    json.dump(result, f, indent=2)

try:
    DRIVE_MODELS  = '/content/drive/MyDrive/ai-recruiter/models'
    DRIVE_REPORTS = '/content/drive/MyDrive/ai-recruiter/reports'
    os.makedirs(DRIVE_MODELS,  exist_ok=True)
    os.makedirs(DRIVE_REPORTS, exist_ok=True)

    dest = f'{DRIVE_MODELS}/{run_name}'
    if os.path.exists(dest): shutil.rmtree(dest)
    shutil.copytree(output_dir, dest)
    shutil.copy(report_path, f'{DRIVE_REPORTS}/{os.path.basename(report_path)}')
    print(f'  Model  → Drive ✓  ({dest})')
    print(f'  Report → Drive ✓  ({DRIVE_REPORTS}/{os.path.basename(report_path)})')
except Exception as e:
    print(f'  Drive upload skipped: {e}')

In [ ]:
# === CELL 6: Summary — Chạy sau khi có đủ kết quả từ Drive ===
# Có thể chạy ở session mới, load report từ Drive
import os, json, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_REPORTS    = '/content/drive/MyDrive/ai-recruiter/reports'
LR_CANDIDATES    = [2e-5, 3e-5, 5e-5]
SEED             = 42
V06_BEST_SINGLE  = 0.6495
V06_ENSEMBLE_ACC = 0.6580
BASE_MODEL       = 'cross-encoder/ms-marco-roberta-base'

results = []
for lr in LR_CANDIDATES:
    report_file = f'{DRIVE_REPORTS}/fine_tune_cross_encoder_v0.7_roberta_lr{lr:.0e}_report.json'
    if not os.path.exists(report_file):
        print(f'Missing: lr={lr:.0e} (chưa chạy hoặc chưa upload Drive)')
        continue
    with open(report_file) as f:
        results.append(json.load(f))

if not results:
    print('Chưa có kết quả nào. Chạy Cell 3-5 trước.')
else:
    best = max(results, key=lambda r: r['test']['LabelAcc'])

    print(f'=== Phase 1b Summary — {BASE_MODEL} ===')
    print(f'{"LR":<10} {"Val Acc":>10} {"Test Acc":>10} {"MAE":>8} {"vs v0.6 single":>16}')
    print('-' * 60)
    for r in results:
        vs   = (r['test']['LabelAcc'] - V06_BEST_SINGLE) * 100
        mark = '  <- best' if r['learning_rate'] == best['learning_rate'] else ''
        print(f'{r["learning_rate"]:<10.0e} {r["val"]["LabelAcc"]*100:>9.2f}%  {r["test"]["LabelAcc"]*100:>9.2f}%  {r["test"]["MAE"]:>7.2f}  {vs:>+12.2f}pp{mark}')

    print(f'\n  v0.6 best single:  {V06_BEST_SINGLE*100:.2f}%')
    print(f'  v0.6 ensemble:     {V06_ENSEMBLE_ACC*100:.2f}%')
    print(f'\n  Best LR: {best["learning_rate"]:.0e}  ->  Test {best["test"]["LabelAcc"]*100:.2f}%  ({(best["test"]["LabelAcc"]-V06_BEST_SINGLE)*100:+.2f}pp)')
    print(f'\n  -> Dung LR={best["learning_rate"]:.0e} cho Phase 2 Ensemble notebook')

    os.makedirs('artifacts/reports', exist_ok=True)
    report_path = 'artifacts/reports/fine_tune_cross_encoder_v0.7_roberta_lr_grid_search_report.json'
    with open(report_path, 'w') as f:
        json.dump({
            'experiment':  'Phase 1b: LR Grid Search (RoBERTa-base)',
            'base_model':  BASE_MODEL,
            'dataset':     'v0.5',
            'seed':        SEED,
            'epochs':      results[0].get('epochs', 10),
            'batch_size':  16,
            'results':     results,
            'best': {
                'learning_rate':  best['learning_rate'],
                'test_labelacc':  best['test']['LabelAcc'],
                'improvement_pp': (best['test']['LabelAcc'] - V06_BEST_SINGLE) * 100,
            },
        }, f, indent=2)

    try:
        shutil.copy(report_path, f'{DRIVE_REPORTS}/fine_tune_cross_encoder_v0.7_roberta_lr_grid_search_report.json')
        print(f'\n  Combined report saved to Drive ✓')
    except Exception as e:
        print(f'  Drive upload skipped: {e}')